<a href="https://colab.research.google.com/github/bindumadamanchi3/ai-upskilling-journey/blob/main/Python_plus_pandas_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# list comprehension: You have a list of AI models with their token limits. Use a list comprehension to return only models with context window above 50000,
# with the context window converted to thousands (divide by 1000).
models = [
    ('GPT-4o', 128000),
    ('GPT-3.5', 16000),
    ('Claude 3.5', 200000),
    ('LLaMA 3', 8000),
    ('Gemini 1.5', 1000000),
    ('Mistral', 32000)
]

# Expected: [('GPT-4o', 128.0), ('Claude 3.5', 200.0), ('Gemini 1.5', 1000.0)]
filter_models = [(i, j/1000) for i,j in models if j > 50000]
print(filter_models)

[('GPT-4o', 128.0), ('Claude 3.5', 200.0), ('Gemini 1.5', 1000.0)]


In [ ]:
#Dict Comprehension - Given a list of API endpoint names and their average response times, build a dict mapping endpoint to response time — but only include endpoints slower than 200ms.

endpoints = [
    ('POST /chat', 450),
    ('GET /health', 12),
    ('POST /embed', 180),
    ('POST /rag/query', 820),
    ('GET /models', 95)
]

# Expected: {'POST /chat': 450, 'POST /rag/query': 820}
result = dict([(i,j) for i,j in endpoints if j > 200])
print(result)

#OR

result1 = {key:value for key,value in endpoints if value > 200}
print(result1)

{'POST /chat': 450, 'POST /rag/query': 820}
{'POST /chat': 450, 'POST /rag/query': 820}


In [ ]:
# Nested Comprehension - You have a list of RAG pipeline stages, each with a list of sub-steps. Flatten everything into a single list of strings in the format "stage: step".
pipeline = [
    ('ingest',    ['load', 'chunk', 'embed']),
    ('retrieve',  ['query', 'rank', 'filter']),
    ('generate',  ['prompt', 'call_llm', 'parse'])
]

# Expected:
# ['ingest: load', 'ingest: chunk', 'ingest: embed',
#  'retrieve: query', 'retrieve: rank', 'retrieve: filter',
#  'generate: prompt', 'generate: call_llm', 'generate: parse']

result = [f"{i}:{k}" for i, j in pipeline for k in j]
print(result)

['ingest:load', 'ingest:chunk', 'ingest:embed', 'retrieve:query', 'retrieve:rank', 'retrieve:filter', 'generate:prompt', 'generate:call_llm', 'generate:parse']


In [ ]:
#Basic Generator - Write a generator llm_cost_stream(calls) that takes a list of (model, tokens) tuples and yields the estimated cost for each call one at a time. Use these rates: gpt4: $0.03/1k tokens, claude: $0.015/1k tokens, default: $0.01/1k tokens.

def llm_cost_stream(calls):
    # for model, tokens in calls:
    #   if model.upper() == "GPT4":
    #     yield (tokens/1000) * 0.03
    #   elif model.upper() == "CLAUDE":
    #     yield (tokens/1000) * 0.015
    #   else:
    #     yield (tokens/1000) * 0.01

    #Above code works but following is more of pythonic way suggested to follow
    cost = {'gpt4' : 0.03, 'claude': 0.015}
    for model,tokens in calls:
      rate = cost.get(model, 0.01)
      yield (tokens/1000) * rate


calls = [('gpt4', 2000), ('claude', 5000), ('gpt4', 1000), ('other', 3000)]
for cost in llm_cost_stream(calls):
   print(f"${cost}")


# Should yield: 0.06, 0.075, 0.03, 0.03

$0.06
$0.075
$0.03
$0.03


In [ ]:
# chained generators - Write two generators:
# read_prompts(prompts) — yields each prompt one at a time
# validate_prompts(prompts, max_len) — wraps the first, only yields prompts shorter than max_len characters, and prints a warning for skipped ones
def read_prompts(prompts):
  for prompt in prompts:
     yield prompt

def validate_prompts(prompts, max_len):
  for prompt in read_prompts(prompts):
    if len(prompt) <= max_len:
      yield prompt
    else:
      print(f"[Skipped]: Length of this prompt:{prompt} : {len(prompt)} exceeds max length: max_len")

prompts = [
    "What is RAG?",
    "Explain the entire history of machine learning from 1950 to present day in exhaustive detail",
    "How do embeddings work?",
    "What is a vector database?",
    "Write a 10000 word essay on transformers"
]

for prompt in validate_prompts(prompts, max_len = 50):
  print(prompt)

# validate_prompts(prompts, max_len=50) should yield 3 prompts, skip 2

What is RAG?
[Skipped]: Length of this prompt:Explain the entire history of machine learning from 1950 to present day in exhaustive detail : 92 exceeds max length: max_len
How do embeddings work?
What is a vector database?
Write a 10000 word essay on transformers


In [ ]:
#Takes a system message
# Takes any number of example strings (few-shot examples)
# Takes keyword config options
# Returns a formatted dict ready to send to an LLM API

def build_prompt(system, *examples, **config):
  # result = {}
  # result['system'] = system
  # result['examples'] = examples
  # return result | config

  #above code works, but following is the more pythonic way suggested to follow
  defaults = {'temperature': 90, 'max_tokens': 1000}
  return {
      'system': system,
      'examples': list(examples),
      # 'temperature': config.get('temperature', 90),
      # 'max_tokens': config.get('max_tokens', 1000) these 2 lines will work too instead of below 2 lines but below is more of a pythonic way
      **defaults,
      **config #this will override defaults
  }

result = build_prompt(
    "You are a helpful assistant.",
    "Q: What is AI? A: Artificial Intelligence.",
    "Q: What is ML? A: Machine Learning.",
    temperature=0.5,
    max_tokens=200
)

for key, value in result.items():
    print(f"{key} : {value}")

# Expected keys: system, examples (list), temperature, max_tokens

system : You are a helpful assistant.
examples : ['Q: What is AI? A: Artificial Intelligence.', 'Q: What is ML? A: Machine Learning.']
temperature : 0.5
max_tokens : 200


In [ ]:
#pandas
# Create this DataFrame, then: check for nulls, fill missing salaries with the median, add a cost_per_token column (salary / 1,000,000).
import pandas as pd

data = {
    'model':    ['GPT-4', 'Claude', 'Gemini', 'LLaMA', 'Mistral'],
    'provider': ['OpenAI', 'Anthropic', 'Google', None, 'Mistral AI'],
    'salary':   [None, 95000, 110000, 75000, 88000],
    'tokens_b': [1.2, 0.8, 1.5, 2.1, 0.6]
}

df = pd.DataFrame(data)

print(df.isnull().sum())

salary_median = df['salary'].median()
df.fillna({'salary' : salary_median}, inplace = True)  # OR df['salary'] = df['salary'].fillna(df['salary'].median())
df.fillna({'provider' : "UnKnown"}, inplace = True)
print(df.isnull().sum())

df['cost_per_token'] = (df['salary']/1000000).round(4)
print(df.info())

print(df)

model       0
provider    1
salary      1
tokens_b    0
dtype: int64
model       0
provider    0
salary      0
tokens_b    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   model           5 non-null      object 
 1   provider        5 non-null      object 
 2   salary          5 non-null      float64
 3   tokens_b        5 non-null      float64
 4   cost_per_token  5 non-null      float64
dtypes: float64(3), object(2)
memory usage: 332.0+ bytes
None
     model    provider    salary  tokens_b  cost_per_token
0    GPT-4      OpenAI   91500.0       1.2          0.0915
1   Claude   Anthropic   95000.0       0.8          0.0950
2   Gemini      Google  110000.0       1.5          0.1100
3    LLaMA     UnKnown   75000.0       2.1          0.0750
4  Mistral  Mistral AI   88000.0       0.6          0.0880


In [ ]:
import pandas as pd
import io

csv_data = '''name,age,role,salary,experience_years,city,passed_screening
Naga,32,ML Engineer,120000,7,Austin,True
Alice,28,Data Engineer,95000,4,New York,True
Bob,35,Backend Engineer,110000,10,San Francisco,False
Carol,26,AI Engineer,105000,3,Austin,True
Dave,40,MLOps Engineer,130000,15,Seattle,True
Eve,30,Data Scientist,98000,6,New York,True
Frank,24,Junior ML Engineer,75000,1,Austin,False
Grace,33,Backend Engineer,112000,9,San Francisco,True
Henry,29,Data Engineer,93000,5,Seattle,False
Iris,27,AI Engineer,108000,4,New York,True'''

engineer_df = pd.read_csv(io.StringIO(csv_data))
print(engineer_df.shape)    # (10, 7)
print(engineer_df.head())

(10, 7)
    name  age              role  salary  experience_years           city  \
0   Naga   32       ML Engineer  120000                 7         Austin   
1  Alice   28     Data Engineer   95000                 4       New York   
2    Bob   35  Backend Engineer  110000                10  San Francisco   
3  Carol   26       AI Engineer  105000                 3         Austin   
4   Dave   40    MLOps Engineer  130000                15        Seattle   

   passed_screening  
0              True  
1              True  
2             False  
3              True  
4              True  


In [ ]:
#Filter with multiple conditions (Medium)
# Using the engineers dataset, in a single chained expression:

# Filter to engineers who passed screening
# AND salary above 100000
# AND city is NOT Austin
# Then sort by salary descending
# Then show only name, city, salary
df = engineer_df
result = df[(df['passed_screening'] == True)
             & (df['salary'] > 100000)
             & ~(df['city'] == "Austin")].sort_values('salary', ascending = False)[['name', 'city', 'salary']]


print(result)

# Expected: Grace, Naga excluded (Austin), Dave, Bob excluded (not passed)
# Remaining should be: Dave(passed=True, Seattle, 130k),
#                      Grace(passed=True, SF, 112k),
#                      Iris(passed=True, NY, 108k)

    name           city  salary
4   Dave        Seattle  130000
7  Grace  San Francisco  112000
9   Iris       New York  108000


In [ ]:
#groupby + multiple aggregations (Medium)
# Group the engineers dataset by city and compute: count of engineers, min salary, max salary, and average experience. Sort by average salary desc

result = engineer_df.groupby('city').agg(
    count = ('name', 'count'),
    min_salary = ('salary', 'min'),
    max_salary = ('salary', 'max'),
    avg_salary = ('salary', 'mean'),
    avg_experience = ('experience_years', 'mean')
).round(1)

print(result.sort_values('avg_salary', ascending = False))


               count  min_salary  max_salary  avg_salary  avg_experience
city                                                                    
Seattle            2       93000      130000    111500.0            10.0
San Francisco      2      110000      112000    111000.0             9.5
New York           3       95000      108000    100333.3             4.7
Austin             3       75000      120000    100000.0             3.7


In [ ]:
#Add columns + filter + sort pipeline (Medium)
# In one pipeline:
# Add seniority (Senior if years >= 7, else Junior)
# Add salary_band (High if salary >= 110000 else Standard)
# Filter to Senior engineers only
# Sort by salary descending
# Print name, seniority, salary_band, salary
import numpy as np

result = (
    engineer_df
    .assign(
        seniority   = np.where(engineer_df['experience_years'] >= 7, 'Senior', 'Junior'),
        salary_band = np.where(engineer_df['salary'] >= 110000, 'High', 'Standard')
    )
    .query("seniority == 'Senior'")
    .sort_values('salary', ascending=False)
    [['name', 'seniority', 'salary_band', 'salary']]
)

print(result)

    name seniority salary_band  salary
4   Dave    Senior        High  130000
0   Naga    Senior        High  120000
7  Grace    Senior        High  112000
2    Bob    Senior        High  110000


In [ ]:
#COMBINED CHALLENGE (Hard)
# Build a Python function analyse_engineers(csv_string) that:

# Loads the CSV from a string
# Adds seniority, salary_band, and salary_monthly columns
# Returns a summary dict with:

# total_engineers — total count
# senior_count — count of Senior engineers
# top_earner — name of highest paid engineer
# avg_salary_by_city — dict of city → average salary (rounded to 0 dp)
# high_band_pct — percentage of engineers in High salary band (rounded to 1 dp)
import pandas as pd

csv_data = '''name,age,role,salary,experience_years,city,passed_screening
Naga,32,ML Engineer,120000,7,Austin,True
Alice,28,Data Engineer,95000,4,New York,True
Bob,35,Backend Engineer,110000,10,San Francisco,False
Carol,26,AI Engineer,105000,3,Austin,True
Dave,40,MLOps Engineer,130000,15,Seattle,True
Eve,30,Data Scientist,98000,6,New York,True
Frank,24,Junior ML Engineer,75000,1,Austin,False
Grace,33,Backend Engineer,112000,9,San Francisco,True
Henry,29,Data Engineer,93000,5,Seattle,False
Iris,27,AI Engineer,108000,4,New York,True'''

def analyse_engineers(csv_string):
  eng_df = pd.read_csv(io.StringIO(csv_string))
  eng_df['seniority'] = np.where(eng_df['experience_years'] >= 7, 'Senior', 'Junior')
  eng_df['salary_band'] = np.where(eng_df['salary'] >= 110000, 'High', 'Standard')
  eng_df['salary_monthly'] = (eng_df['salary']/12).round(2)

  return {
      'total_engineers': len(eng_df),
      'senior_count': int((eng_df['seniority'] == "Senior").sum()),
      'top_earner': eng_df.loc[eng_df['salary'].idxmax(), 'name'],
      'avg_salary_by_city' : eng_df.groupby('city')['salary'].mean().round(0).astype(int).to_dict(),
      'high_band_pct': round((eng_df['salary_band'] == "High").mean() *100, 1)
  }

result = analyse_engineers(csv_data)
print(result)
print(result['top_earner'])          # Dave
print(result['senior_count'])        # 4
print(result['high_band_pct'])       # 40.0



{'total_engineers': 10, 'senior_count': 4, 'top_earner': 'Dave', 'avg_salary_by_city': {'Austin': 100000, 'New York': 100333, 'San Francisco': 111000, 'Seattle': 111500}, 'high_band_pct': np.float64(40.0)}
Dave
4
40.0


In [ ]:
#Basic decorator (Easy)
# Write a @validate_prompt decorator that checks if the first argument of a function is a non-empty string. If it's empty or not a string, print an error and return None without calling the function.
from functools import wraps

def validate_prompt(func):
  @wraps(func)
  def wrapper(*args, **kwargs):
     prompt = args[0] if args else kwargs.get('prompt', '')
     if not isinstance(prompt, str) or not prompt.strip():
        print("First argument must be a string and cannot be empty")
        return None
     return func(*args, **kwargs)
  return wrapper

@validate_prompt
def call_llm(prompt, model='claude'):
    return f"Response to: {prompt}"

print(call_llm("What is RAG?"))     # should work normally
print(call_llm(""))                  # should print error, return None
print(call_llm(None))                # should print error, return None

Response to: What is RAG?
First argument must be a string and cannot be empty
None
First argument must be a string and cannot be empty
None


In [ ]:
#Decorator factory (Medium)
#Write a @rate_limit(calls_per_minute) decorator factory that tracks how many times a function has been called and prints a warning when
# it exceeds the limit. For simplicity, just count total calls (no time window needed).
from functools import wraps

def rate_limit(calls_per_minute):
  def decorator(func):
    counter = 0  #call_count = [0]   # list so inner function can mutate it
    @wraps(func)
    def wrapper(*args, **kwargs):
      nonlocal counter    #Note the call_count = [0] list trick — a nonlocal int can't be mutated inside a nested function without nonlocal. A list works because you're mutating its contents, not rebinding it.
      counter += 1   #call_count[0] += 1
      if counter > calls_per_minute:
       print(f'[WARNING] Rate limit exceeded: call #{counter} '
                      f'(limit: {calls_per_minute}/min)')
      return func(*args, **kwargs)
    return wrapper
  return decorator

@rate_limit(calls_per_minute=3)
def call_api(prompt):
    return f"Response: {prompt[:20]}"

# First 3 calls: normal
# 4th call onwards: print warning but still execute
call_api("Hello!")
call_api("Hello!")
call_api("Hello!")
call_api("Hello!")
call_api("Hello!")

[WARNING] Rate limit exceeded: call #4 (limit: 3/min)
[WARNING] Rate limit exceeded: call #5 (limit: 3/min)


'Response: Hello!'

In [ ]:
# Context manager (Medium)
# Write a @contextmanager called llm_session(model) that:

# Prints "Opening session: {model}" on entry
# Yields a dict the caller can write results into
# On exit prints "Closing session — {n} calls made" using the count stored in the dict

# When you put @contextmanager over a function, Python transforms that function into a special object.

# The decorator internally calls the generator and gets the "iterator" ready.

# When you hit the with line, the decorator automatically calls next() the first time. This runs your code up to the yield.

# When the code inside the with block finishes, the decorator automatically calls next() a second time. This "resumes" your function from where it paused.

# how many times it is called ? In a standard context manager like your llm_session, it happens exactly once per with block.Entry: next() is called $\rightarrow$ Runs until yield.Execution: Your code inside the with block runs.Exit: next() is called again $\rightarrow$ Runs from the yield to the end of the function.If you try to call next() a third time, the generator would raise a StopIteration error, but the decorator handles that gracefully and closes the context.

from contextlib import contextmanager

@contextmanager
def llm_session(model):
  print(f"Opening session: {model}")
  session = {}
  yield session
  calls = session.get('calls', 0)
  tokens = session.get('tokens', 0)
  print(f"Closing session — {calls} calls made | {tokens} tokens used")

with llm_session('claude-sonnet') as session:
    session['calls'] = 4
    session['tokens'] = 8420

# Opening session: claude-sonnet
# Closing session — 4 calls made | 8420 tokens used


Opening session: claude-sonnet
Closing session — 4 calls made | 8420 tokens used


In [2]:
import pandas as pd
import numpy as np
import io
from dataclasses import dataclass
from typing import Optional
from functools import wraps
from contextlib import contextmanager

# Dataset — LLM API call logs
csv_data = '''call_id,model,prompt_tokens,completion_tokens,latency_ms,status,user_id,cost_usd
1,gpt4,250,180,420,success,u001,0.013
2,claude,800,450,890,success,u002,0.019
3,gpt4,120,0,5100,timeout,u001,0.0
4,claude,600,380,760,success,u003,0.015
5,gpt35,300,200,210,success,u002,0.005
6,gpt4,950,600,1100,success,u004,0.046
7,claude,200,150,340,success,u001,0.005
8,gpt35,450,300,280,success,u003,0.008
9,gpt4,1200,0,6200,timeout,u002,0.0
10,claude,500,320,650,success,u004,0.012
11,gpt35,280,190,195,success,u001,0.005
12,gpt4,750,480,980,success,u003,0.037
13,claude,900,550,1050,success,u002,0.022
14,gpt35,150,100,180,success,u004,0.003
15,gpt4,600,400,870,success,u001,0.029'''

df = pd.read_csv(io.StringIO(csv_data))
print(df.shape)    # (15, 8)
print(df.head())

(15, 8)
   call_id   model  prompt_tokens  completion_tokens  latency_ms   status  \
0        1    gpt4            250                180         420  success   
1        2  claude            800                450         890  success   
2        3    gpt4            120                  0        5100  timeout   
3        4  claude            600                380         760  success   
4        5   gpt35            300                200         210  success   

  user_id  cost_usd  
0    u001     0.013  
1    u002     0.019  
2    u001     0.000  
3    u003     0.015  
4    u002     0.005  


In [ ]:
#Extract and transform (Easy)
#From the call logs, extract a list of call_id values where the call timed out AND prompt_tokens was above 500. Return them as strings in the format "call_{id}"

calls = [
    (1, 250, 'success'), (3, 120, 'timeout'),
    (6, 950, 'success'), (9, 1200, 'timeout'),
    (12, 750, 'success'), (15, 600, 'success')
]

# Expected: ['call_9']   (only call 9: timeout AND tokens > 500)

result = [f"call_{call_id}" for call_id, tokens, status in calls if tokens > 500 and status == "timeout"]

print(result)


['call_9']


In [ ]:
#Dict comprehension from two lists (Easy)
# You have two lists — model names and their average latencies. Build a dict mapping model → latency, but only for models with latency under 1000ms. Then build a second dict with latency converted to seconds (rounded to 3 dp).

models    = ['gpt4', 'claude', 'gpt35', 'llama', 'gemini']
latencies = [870, 740, 220, 1450, 980]

fast = {m:l for m,l in zip(models, latencies) if l < 1000}
print(fast)
# {'gpt4': 870, 'claude': 740, 'gpt35': 220, 'gemini': 980}

in_seconds = {m:(l/1000) for m,l in fast.items()}
print(in_seconds)
# {'gpt4': 0.87, 'claude': 0.74, 'gpt35': 0.22, 'gemini': 0.98}

{'gpt4': 870, 'claude': 740, 'gpt35': 220, 'gemini': 980}
{'gpt4': 0.87, 'claude': 0.74, 'gpt35': 0.22, 'gemini': 0.98}


In [ ]:
#Nested comprehension (Medium)
#You have a list of users, each with a list of models they've used. Build a flat list of "user:model" strings, but only for models that start with 'gpt'.

user_models = [
    ('u001', ['gpt4', 'claude', 'gpt35']),
    ('u002', ['claude', 'gemini']),
    ('u003', ['gpt4', 'gpt35', 'llama']),
    ('u004', ['gpt35'])
]

# Expected:
# ['u001:gpt4', 'u001:gpt35', 'u003:gpt4', 'u003:gpt35', 'u004:gpt35']

result = [f"{user}:{model}"
          for user, models in user_models
          for model in models if model.startswith('gpt')] #'gpt' in model
print(result)


['u001:gpt4', 'u001:gpt35', 'u003:gpt4', 'u003:gpt35', 'u004:gpt35']


In [ ]:
#Set and invert a dict (Medium)
# Given a dict of model → provider, build:

# A set of unique providers
# An inverted dict of provider → list of models they offer

model_provider = {
    'gpt4':    'OpenAI',
    'gpt35':   'OpenAI',
    'claude':  'Anthropic',
    'gemini':  'Google',
    'llama':   'Meta',
    'mistral': 'Mistral AI'
}

unique_providers = {p for p in model_provider.values()}
print(unique_providers)

inverted = {}
for model, provider in model_provider.items():
    inverted.setdefault(provider, []).append(model)
print(inverted)

# Expected providers set: {'OpenAI', 'Anthropic', 'Google', 'Meta', 'Mistral AI'}
# Expected inverted: {'OpenAI': ['gpt4', 'gpt35'], 'Anthropic': ['claude'], ...}

{'Meta', 'Google', 'Mistral AI', 'Anthropic', 'OpenAI'}
{'OpenAI': ['gpt4', 'gpt35'], 'Anthropic': ['claude'], 'Google': ['gemini'], 'Meta': ['llama'], 'Mistral AI': ['mistral']}


In [ ]:
#Generator with running total (Easy)
# Write a generator running_cost(calls) that takes a list of (call_id, cost) tuples and yields (call_id, cost, running_total) — the cumulative cost up to that call

calls = [(1, 0.013), (2, 0.019), (3, 0.0), (4, 0.015), (5, 0.005)]

def running_cost(calls):
  total = 0.0
  for callid,cost in calls:
    total+=cost
    yield (callid, cost, total)

for cost in running_cost(calls):
  print(cost)

# Should yield:
# (1, 0.013, 0.013)
# (2, 0.019, 0.032)
# (3, 0.0,   0.032)
# (4, 0.015, 0.047)
# (5, 0.005, 0.052)

(1, 0.013, 0.013)
(2, 0.019, 0.032)
(3, 0.0, 0.032)
(4, 0.015, 0.047)
(5, 0.005, 0.052)


In [ ]:
#Generator pipeline (Medium)
# Build a 3-stage generator pipeline:

# stream_calls(records) — yields each record one at a time
# filter_success(records) — wraps stage 1, only yields successful calls
# enrich(records) — wraps stage 2, adds a total_tokens field (prompt + completion) to each record before yielding

def stream_calls(records):
  for record in records:
    yield record

def filter_success(records):
  for record in stream_calls(records):
    if record.get('status', '') == 'success':
      yield record

def enrich(records):
  for record in filter_success(records):
    record['total_tokens'] = record.get('prompt_tokens', 0) + record.get('completion_tokens', 0)
    yield record

records = [
    {'id': 1, 'status': 'success', 'prompt_tokens': 250, 'completion_tokens': 180},
    {'id': 2, 'status': 'timeout', 'prompt_tokens': 120, 'completion_tokens': 0},
    {'id': 3, 'status': 'success', 'prompt_tokens': 600, 'completion_tokens': 380},
]

for record in enrich(records):
  print(record)
# enrich() should yield only the 2 successful calls
# each with a new 'total_tokens' key

{'id': 1, 'status': 'success', 'prompt_tokens': 250, 'completion_tokens': 180, 'total_tokens': 430}
{'id': 3, 'status': 'success', 'prompt_tokens': 600, 'completion_tokens': 380, 'total_tokens': 980}


In [ ]:
# Decorator: Timer + logger combined (Easy)
# Write a single @monitor decorator that both times the function AND logs the result. Print: function name, execution time, and the return value.
from functools import wraps
import time

def monitor(func):
  @wraps(func)
  def wrapper(*args, **kwargs):
    start_time = time.time()
    cost = func(*args, **kwargs)
    elapsed = time.time()-start_time
    print(f"[MONITOR] {func.__name__} | {elapsed:.3f}s | returned: {cost:.3f}")
    return cost
  return wrapper

@monitor
def compute_cost(tokens, rate):
    import time; time.sleep(0.05)
    return round((tokens / 1000) * rate, 4)

compute_cost(2500, 0.03)
# [MONITOR] compute_cost | 0.050s | returned: 0.075

[MONITOR] compute_cost | 0.050s | returned: 0.075


0.075

In [ ]:
# Decorator factory with threshold (Medium)
# Write @alert_if_slow(threshold_ms) that runs the function, measures latency, and prints an alert if it exceeds the threshold. The function should still return its result normally either way.

from functools import wraps
import time

def alert_if_slow(threshold_ms):
  def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
      start_time = time.time()
      result = func(*args, **kwargs)
      latency = (time.time() - start_time) * 1000
      if latency > threshold_ms:
        print(f"[Alert] {func.__name__} took {latency:.0f}ms — exceeds threshold of {threshold_ms}ms")
      return result
    return wrapper
  return decorator

@alert_if_slow(threshold_ms=100)
def call_model(prompt):
    import time
    time.sleep(0.15)   # simulate slow API
    return f"Response to: {prompt}"

result = call_model("What is RAG?")
print(result)
# [ALERT] call_model took 150ms — exceeds threshold of 100ms
# Response to: What is RAG?

[Alert] call_model took 150ms — exceeds threshold of 100ms
Response to: What is RAG?


In [ ]:
#args/kwargs + type hints
#  Flexible API caller (Medium)
# Write api_call(endpoint, *payloads, **options) that:

# Accepts an endpoint string
# Accepts any number of payload dicts (merged in order)
# Accepts keyword options (timeout, retries, stream)
# Returns a single merged request dict

def api_call(endpoint, *payloads, **options):
  defaults = {'timeout':30, 'retries': 3, 'stream': True}
  merged_payloads = {}
  for p in payloads:
    merged_payloads.update(p)
  return {
      'endpoint': endpoint,
      **merged_payloads,
      **defaults,
      **options #defaults will be overrided when the value is passed, else show defaults
  }

result = api_call(
    '/v1/chat',
    {'model': 'claude'},
    {'temperature': 0.7},
    timeout=30,
    stream=True
)

print(result)
# {'endpoint': '/v1/chat', 'model': 'claude', 'temperature': 0.7,
#  'timeout': 30, 'retries': 3, 'stream': True}

{'endpoint': '/v1/chat', 'model': 'claude', 'temperature': 0.7, 'timeout': 30, 'retries': 3, 'stream': True}


In [ ]:
#Typed dataclass with methods (Medium)
# Create a @dataclass called APICall with fields: call_id (int), model (str), prompt_tokens (int), completion_tokens (int), latency_ms (float), status (str), cost_usd (float).
# Add two methods:

# total_tokens() — returns prompt + completion tokens
# is_slow(threshold_ms=500) — returns True if latency exceeds threshold

from dataclasses import dataclass

@dataclass
class APICall:
  call_id : int
  model : str
  prompt_tokens : int
  completion_tokens : int
  latency_ms : float
  status : str
  cost_usd : float

  def total_tokens(self):
    return self.prompt_tokens + self.completion_tokens

  def is_slow(self, threshold_ms=500):
    return self.latency_ms > threshold_ms

call = APICall(1, 'gpt4', 250, 180, 870.0, 'success', 0.013)
print(call.total_tokens())       # 430
print(call.is_slow())            # True  (870 > 500)
print(call.is_slow(1000))        # False (870 < 1000)

430
True
False


In [ ]:
#Pandas - Inspect and enrich the call logs (Easy)

# Using the API call logs dataset:

# Print shape, dtypes, and null counts
# Add total_tokens (prompt + completion)
# Add cost_per_token (cost_usd / total_tokens, rounded to 6dp — handle divide by zero for timeouts)
# Add is_slow (True if latency_ms > 500)
import pandas as pd

#print(df)
print("shape", df.shape)
print("dtypes", df.dtypes)
print("Null Counts", df.isnull().sum())

df['total_tokens'] = df['prompt_tokens'] + df['completion_tokens']


df['cost_per_token'] = np.where(df['total_tokens'] >0,  (df['cost_usd']/df['total_tokens']).round(6), 0.0)

df['slow'] = df['latency_ms'] > 500

print(df)



shape (15, 11)
dtypes call_id                int64
model                 object
prompt_tokens          int64
completion_tokens      int64
latency_ms             int64
status                object
user_id               object
cost_usd             float64
total_tokens           int64
cost_per_token       float64
slow                    bool
dtype: object
Null Counts call_id              0
model                0
prompt_tokens        0
completion_tokens    0
latency_ms           0
status               0
user_id              0
cost_usd             0
total_tokens         0
cost_per_token       0
slow                 0
dtype: int64
    call_id   model  prompt_tokens  completion_tokens  latency_ms   status  \
0         1    gpt4            250                180         420  success   
1         2  claude            800                450         890  success   
2         3    gpt4            120                  0        5100  timeout   
3         4  claude            600                380  

In [ ]:
# Filter and analyse failures (Medium)
# From the call logs:
# 14a. Get all failed (timeout) calls — show call_id, model, prompt_tokens, latency_ms
# 14b. What percentage of calls timed out? (rounded to 1dp)
# 14c. Among successful calls only, which model has the highest average latency?
# 14d. Get all calls where latency > 800ms AND status is success AND cost > 0.01
import numpy as np

timed_out = df[df['status'] == "timeout"][['call_id', 'model', 'prompt_tokens', 'latency_ms']]
print(timed_out)

percentage_timedout = round(((df['status'] == "timeout").mean() * 100), 1)
print(f"Timeout rate: {percentage_timedout}%")

success_calls = df[df['status'] == "success"]
avg_latency_by_model = success_calls.groupby('model')['latency_ms'].mean()
print(avg_latency_by_model.sort_values(ascending = False))

slow_and_success_calls = df.query('latency_ms > 800 and status=="success" and cost_usd > 0.01')
print(slow_and_success_calls)

   call_id model  prompt_tokens  latency_ms
2        3  gpt4            120        5100
8        9  gpt4           1200        6200
Timeout rate: 13.3%
model
gpt4      842.50
claude    738.00
gpt35     216.25
Name: latency_ms, dtype: float64
    call_id   model  prompt_tokens  completion_tokens  latency_ms   status  \
1         2  claude            800                450         890  success   
5         6    gpt4            950                600        1100  success   
11       12    gpt4            750                480         980  success   
12       13  claude            900                550        1050  success   
14       15    gpt4            600                400         870  success   

   user_id  cost_usd  total_tokens  cost_per_token  slow  
1     u002     0.019          1250        0.000015  True  
5     u004     0.046          1550        0.000030  True  
11    u003     0.037          1230        0.000030  True  
12    u002     0.022          1450        0.000015  T

In [ ]:
#groupby + pivot analysis (Medium)
# From the call logs:
# 15a. For each model, compute: total calls, success rate (%), average latency (success only), total cost
# 15b. Which user has spent the most in total?
# 15c. For each user, show how many calls they made to each model (pivot-style using groupby)

call_info = df.groupby('model').agg(
    total_calls = ('call_id', 'size'),
    total_cost = ('cost_usd', 'sum')
)
print(call_info)

df['is_success'] = df['status'] == 'success'

model_stats = df.groupby('model').agg(
    total_calls=('status', 'count'),
    success_count=('is_success', 'sum'),
    avg_latency_success=('latency_ms', lambda x: x[df.loc[x.index, 'status'] == 'success'].mean())
)
print(model_stats)

model_stats['success_rate_%'] = (model_stats['success_count'] / model_stats['total_calls']) * 100

print(model_stats[['success_rate_%', 'avg_latency_success']])

# OR
# # Success rate separately
# success_rate = (df.groupby('model')['status']
#                 .apply(lambda x: (x == 'success').mean() * 100)
#                 .round(1))
# model_summary['success_rate_pct'] = success_rate

# # Avg latency on successful calls only
# avg_lat = (df[df['status'] == 'success']
#            .groupby('model')['latency_ms']
#            .mean().round(0))
# model_summary['avg_latency_success'] = avg_lat

# print(model_summary.sort_values('total_cost', ascending=False))

#which user has spent the most the total
max_spent_user = df.groupby('user_id')['cost_usd'].sum().round(3)
print("User's that spent the most the total:")
print(max_spent_user)

#For each user, show how many calls they made to each model (pivot-style using groupby) #try with and without unstack to view the oytput display
calls_to_model = df.groupby(['user_id','model'])['model'].count().unstack()
print('\nCalls per user per model:')
print(calls_to_model)


        total_calls  total_cost
model                          
claude            5       0.073
gpt35             4       0.021
gpt4              6       0.125
        total_calls  success_count  avg_latency_success
model                                                  
claude            5              5               738.00
gpt35             4              4               216.25
gpt4              6              4               842.50
        success_rate_%  avg_latency_success
model                                      
claude      100.000000               738.00
gpt35       100.000000               216.25
gpt4         66.666667               842.50
User's that spent the most the total:
user_id
u001    0.052
u002    0.046
u003    0.060
u004    0.061
Name: cost_usd, dtype: float64

Calls per user per model:
model    claude  gpt35  gpt4
user_id                     
u001          1      1     3
u002          2      1     1
u003          1      1     1
u004          1      1     1


In [5]:
# COMBINED CHALLENGE (Hard) - Build api_log_report(csv_string) — a function that returns a full report dict:

# total_calls — total number of calls
# timeout_rate_pct — percentage that timed out (1dp)
# total_cost_usd — total spend across all calls (4dp)
# most_used_model — model with the highest call count
# most_expensive_call — dict with call_id, model, cost_usd of the single most expensive call
# slow_success_calls — list of call_ids where status=success AND latency > 800ms
# cost_by_model — dict of model → total cost (rounded to 4dp)

def api_log_report():

  expensive_call = df.loc[df['cost_usd'].idxmax()]
  most_expensive_call = {
      'callid' : expensive_call['call_id'],
      'model' : expensive_call['model'],
      'cost_usd': expensive_call['cost_usd']
  }

  # Most expensive call
    # idx = df['cost_usd'].idxmax()
    # top_call = df.loc[idx, ['call_id', 'model', 'cost_usd']].to_dict()

  return {
      'total_calls' : len(df),
      'timeout_rate_pct' :round ((df['status'] == 'timeout').mean() * 100, 1),
      'total_cost_usd' : round(df['cost_usd'].sum(), 4),
      'most_used_model' : (df.groupby('model')['call_id'].count()).idxmax(),  # df['model'].value_counts().idxmax(),
      'most_expensive_call' : most_expensive_call,
      'highest_call_count' : (df.groupby('model')['call_id'].count()).idxmax(),
      'slow_success_calls' : df[(df['status'] == 'success') & (df['latency_ms'] > 800)]['call_id'].tolist(),
      'cost_by_model' : df.groupby('model')['cost_usd'].sum().round(4).to_dict()
  }

report = api_log_report()
print(report)

for k, v in report.items():
    print(f"{k}: {v}")

{'total_calls': 15, 'timeout_rate_pct': np.float64(13.3), 'total_cost_usd': np.float64(0.219), 'most_used_model': 'gpt4', 'most_expensive_call': {'callid': np.int64(6), 'model': 'gpt4', 'cost_usd': np.float64(0.046)}, 'highest_call_count': 'gpt4', 'slow_success_calls': [2, 6, 12, 13, 15], 'cost_by_model': {'claude': 0.073, 'gpt35': 0.021, 'gpt4': 0.125}}
total_calls: 15
timeout_rate_pct: 13.3
total_cost_usd: 0.219
most_used_model: gpt4
most_expensive_call: {'callid': np.int64(6), 'model': 'gpt4', 'cost_usd': np.float64(0.046)}
highest_call_count: gpt4
slow_success_calls: [2, 6, 12, 13, 15]
cost_by_model: {'claude': 0.073, 'gpt35': 0.021, 'gpt4': 0.125}


In [23]:
import pandas as pd
import numpy as np

# Mock Data Setup
data = {
    'provider': ['OpenAI', 'OpenAI', 'Anthropic', 'Anthropic', 'Google', 'OpenAI'],
    'model': ['gpt-4o', 'gpt-4o', 'claude-3-sonnet', 'claude-3-sonnet', 'gemini-1.5', 'gpt-4o'],
    'latency_ms': [1200, 850, 2100, 1900, 400, 3100],
    'status': ['success', 'success', 'success', 'success', 'success', 'success'],
    'tokens': [25000, 30000, 15000, 40000, 80000, 5000]
}
new_df = pd.DataFrame(data)
print(new_df)

    provider            model  latency_ms   status  tokens
0     OpenAI           gpt-4o        1200  success   25000
1     OpenAI           gpt-4o         850  success   30000
2  Anthropic  claude-3-sonnet        2100  success   15000
3  Anthropic  claude-3-sonnet        1900  success   40000
4     Google       gemini-1.5         400  success   80000
5     OpenAI           gpt-4o        3100  success    5000


In [24]:
#Filter the data to include only successful calls (status == 'success') from the providers 'OpenAI' and 'Anthropic'.
filtered_calls = new_df[(new_df['provider'].isin(['OpenAI', 'Anthropic'])) & (new_df['status'] == "success")]

#Group by both provider and model.For each group, calculate the total tokens processed and the 95th percentile latency (use lambda x: x.quantile(0.95)).
group_by = filtered_calls.groupby(['provider', 'model']).agg(
    total_tokens = ('tokens', 'sum'),
    p95_latency = ('latency_ms', lambda x: x.quantile(0.95))
).reset_index()

#Filter out any model that processed fewer than 50,000 total tokens
filter_tokens = group_by[group_by['total_tokens'] >= 50000]
print(filter_tokens)

#Sort the final results by the 95th percentile latency in descending order so the slowest models at the tail-end appear first.
final_result = filter_tokens.sort_values('p95_latency', ascending = False)
print(final_result)


    provider            model  total_tokens  p95_latency
0  Anthropic  claude-3-sonnet         55000       2090.0
1     OpenAI           gpt-4o         60000       2910.0
    provider            model  total_tokens  p95_latency
1     OpenAI           gpt-4o         60000       2910.0
0  Anthropic  claude-3-sonnet         55000       2090.0


In [48]:
import pandas as pd

# Mock Data Setup
cost_df = pd.DataFrame({
    'user_id': [101, 102, 101, 103, 104],
    'organization': ['Acme Corp', 'Stark Industries', 'Acme Corp', 'Stark Industries', 'Acme Corp'],
    'cost_usd': [0.012, 0.085, 0.061, 0.004, 0.092],
    'call_id': [1, 2, 3, 4, 5]
})

#Use the .query() syntax to filter for rows where cost_usd strictly exceeds an externally defined system threshold variable cost_limit = 0.05.
cost_limit = 0.05
cost_limit_filter = cost_df.query('cost_usd > @cost_limit')
print("--- cost_limit >= 0.05 ---")
print(cost_limit_filter)

#Group the remaining data by organization to find the total spend and the maximum single call cost for each organization.
organization_filter = cost_limit_filter.groupby('organization').agg(
    total_spend = ('cost_usd', 'sum'),
    maximum_single_call_cost = ('cost_usd', 'max')
).reset_index()

#Sort the organizations from highest total spend to lowest.
result = organization_filter.sort_values('total_spend', ascending = False)
print("--- Sorted DataFrame ---")
print(result)


--- cost_limit >= 0.05 ---
   user_id      organization  cost_usd  call_id
1      102  Stark Industries     0.085        2
2      101         Acme Corp     0.061        3
4      104         Acme Corp     0.092        5
--- Sorted DataFrame ---
       organization  total_spend  maximum_single_call_cost
0         Acme Corp        0.153                     0.092
1  Stark Industries        0.085                     0.085


In [64]:
import pandas as pd

# Mock Data Setup
router_df = pd.DataFrame({
    'router_id': ['R-01', 'R-01', 'R-02', 'R-01', 'R-02', 'R-02'],
    'endpoint': ['/v1/chat', '/v1/embeddings', '/v1/chat', '/v1/chat', '/v1/chat', '/v1/embeddings'],
    'response_code': [500, 200, 404, 503, 500, 401],
    'retry_count': [2, 0, 1, 3, 2, 0]
})
print(router_df)

#Filter the DataFrame to find problematic calls: rows where the response_code is not 200 AND the retry_count is greater than 0.
problematic_calls = router_df[(router_df['response_code'] != 200) & (router_df['retry_count'] > 0)]

#Group by router_id and endpoint simultaneously. Calculate the count of these problematic calls per group.
groupby_result = problematic_calls.groupby(['router_id', 'endpoint']).agg(
    count = ('response_code', 'count')
).reset_index()
print("--------count of these problematic calls per group.-----")
print(groupby_result)

#Keep the multi-index layout intact (do not use as_index=False), but sort the resulting MultiIndex DataFrame by the call counts in descending order.
result = groupby_result.sort_values('count', ascending = False)
print("-------final result---------")
print(result)

  router_id        endpoint  response_code  retry_count
0      R-01        /v1/chat            500            2
1      R-01  /v1/embeddings            200            0
2      R-02        /v1/chat            404            1
3      R-01        /v1/chat            503            3
4      R-02        /v1/chat            500            2
5      R-02  /v1/embeddings            401            0
--------count of these problematic calls per group.-----
  router_id  endpoint  count
0      R-01  /v1/chat      2
1      R-02  /v1/chat      2
-------final result---------
  router_id  endpoint  count
0      R-01  /v1/chat      2
1      R-02  /v1/chat      2
